# Fase 3 · CoALA
Trabajo: 12 mensajes; episodios: FastEmbed multilingüe; preferencias explícitas; procedimiento: verificación. Mem0 y Letta administran memoria con abstracciones propias; aquí SQLite conserva identidad y versión vectorial.

In [1]:
from pathlib import Path
import os
if Path.cwd().name == 'notebooks': os.chdir('..')

In [2]:
from pedro.memory import (FastEmbedder, Historial, nueva_sesion, guardar_episodio,
                          actualizar_episodio, recuperar, guardar_preferencia, preferencias)
from langchain_core.messages import HumanMessage, AIMessage
ruta = Path('storage/notebooks/memoria.sqlite3')
sesion = nueva_sesion(ruta, 'alumna', 'demo')
historia = Historial(ruta, 'alumna', sesion)
historia.add_messages([HumanMessage(content='Quiero dos piezas'), AIMessage(content='¿Qué plazo?')])
historia.add_messages([HumanMessage(content='Siete días'), AIMessage(content='Entendido')])
guardar_preferencia(ruta, 'alumna', 'proveedor', 'nacional', explicita=True)
emb = FastEmbedder()
eid = guardar_episodio(ruta, 'alumna', sesion, 'prefiere proveedor nacional', emb)
print([m.content for m in historia.todos()], preferencias(ruta, 'alumna'))
print('antes', recuperar(ruta, 'alumna', 'proveedor nacional', emb))
actualizar_episodio(ruta, 'alumna', eid, 'corrigió: acepta proveedor internacional', emb)
print('después', recuperar(ruta, 'alumna', 'proveedor internacional', emb))

/home/efraxpc/infnet/agentes/pedro_asistente_de_compras/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['Quiero dos piezas', '¿Qué plazo?', 'Siete días', 'Entendido'] {'proveedor': 'nacional'}
antes [{'id': 16, 'dataset_id': 'demo', 'texto': 'prefiere proveedor nacional', 'similitud': np.float64(0.6358346006771141)}, {'id': 1, 'dataset_id': 'demo', 'texto': 'corrigió: acepta proveedor internacional', 'similitud': np.float64(0.5200657950799659)}, {'id': 2, 'dataset_id': 'demo', 'texto': 'corrigió: acepta proveedor internacional', 'similitud': np.float64(0.5200657950799659)}]
después [{'id': 1, 'dataset_id': 'demo', 'texto': 'corrigió: acepta proveedor internacional', 'similitud': np.float64(0.6689239159997002)}, {'id': 2, 'dataset_id': 'demo', 'texto': 'corrigió: acepta proveedor internacional', 'similitud': np.float64(0.6689239159997002)}, {'id': 3, 'dataset_id': 'demo', 'texto': 'corrigió: acepta proveedor internacional', 'similitud': np.float64(0.6689239159997002)}]


Precio y stock guardados en episodios son históricos. La petición actual prevalece.

## R13 · mapa de memorias (CoALA)
Ejecución real local, sin modelo. Tras dos turnos de compra y una preferencia, `mapa_memorias` muestra qué hay en cada tipo de memoria. La conversación equivalente con el modelo real está en `docs/trazas/fase3_memoria_llm.json`.

In [3]:
import json
from pedro.agent import crear_flujo
from pedro.catalog import crear_escenario
from pedro.memory import mapa_memorias
escenario = crear_escenario(Path('storage/notebooks/catalogos'))
ruta13 = Path('storage/notebooks/r13.sqlite3')
sesion13 = nueva_sesion(ruta13, 'alumna', escenario.dataset_id)
local = crear_flujo(escenario, ruta13)
for texto in ('Necesito 2 unidades de PEC-01-0001 con plazo 7 días', 'Ahora que sean 3 y mantén el plazo'):
    print(texto, '->', local.invoke({'input': texto}, config={'configurable': {
        'user_id': 'alumna', 'session_id': sesion13}}).content.splitlines()[0])
guardar_preferencia(ruta13, 'alumna', 'proveedor', 'nacionales', explicita=True)
guardar_episodio(ruta13, 'alumna', sesion13, 'pidió 3 unidades de PEC-01-0001 con plazo 7 días', emb)
for memoria_coala, contenido in mapa_memorias(ruta13, 'alumna', sesion13).items():
    print(memoria_coala, '->', json.dumps(contenido, ensure_ascii=False))

Necesito 2 unidades de PEC-01-0001 con plazo 7 días -> CentroPeças: R$ 2.963,07 para 2 unidades; plazo 2 días.
Ahora que sean 3 y mantén el plazo -> CentroPeças: R$ 4.401,99 para 3 unidades; plazo 2 días.
trabajo (corto plazo, por sesión) -> {"borrador_de_compra": {"producto_id": "PEC-01-0001", "cantidad": 3, "plazo_max_dias": 7}, "mensajes_en_la_ventana": 4, "mensajes_guardados": 4}
episódica (largo plazo, por usuario) -> {"episodios": 1, "sesiones_de_origen": 1, "recuperación": "similitud coseno, top 3, modelo sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"}
semántica (largo plazo) -> {"preferencias_del_usuario": {"proveedor": "nacionales"}, "conocimiento_del_dominio": "catálogo SQLite del escenario y políticas en FAISS"}
procedimental (largo plazo) -> {"donde": "prompt PTCF, ejemplos few-shot, reglas de diálogo y código de verificación", "cambia": "solo con una nueva versión del proyecto"}


/tmp/ipykernel_23396/1992851600.py:8: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  local = crear_flujo(escenario, ruta13)


## R16 · chatbot con ConversationBufferMemory
**Doble de prueba:** `FakeListLLM` devuelve respuestas fijas. Lo real es `ConversationChain` con `ConversationBufferMemory`: se imprime el prompt del último turno para ver que lleva la conversación entera. La conversación con el modelo real está en `docs/trazas/fase3_memoria_llm.json`.

In [4]:
from langchain_core.language_models.fake import FakeListLLM
from pedro.legacy import crear_chatbot_buffer
chatbot, buffer = crear_chatbot_buffer(FakeListLLM(responses=[
    'Hola Ana, ¿cuántas unidades necesitas?', 'Anotado: 2 unidades. ¿Presupuesto y plazo?',
    'Ana, pediste 2 termostatos, hasta R$ 900, en 5 días.']))
for texto in ('Hola, soy Ana. Necesito un termostato para mi nevera.', 'Que sean 2 unidades.',
              'Tengo hasta R$ 900 y lo necesito en 5 días. Resume mi solicitud.'):
    print('Usuario:', texto)
    print('Pedro  :', chatbot.invoke({'input': texto})['response'])
    print('   caracteres en memoria:', len(buffer.load_memory_variables({})['history']))
print('--- memoria de buffer ---')
print(buffer.load_memory_variables({})['history'])

Usuario: Hola, soy Ana. Necesito un termostato para mi nevera.
Pedro  : Hola Ana, ¿cuántas unidades necesitas?
   caracteres en memoria: 108
Usuario: Que sean 2 unidades.
Pedro  : Anotado: 2 unidades. ¿Presupuesto y plazo?
   caracteres en memoria: 188
Usuario: Tengo hasta R$ 900 y lo necesito en 5 días. Resume mi solicitud.
Pedro  : Ana, pediste 2 termostatos, hasta R$ 900, en 5 días.
   caracteres en memoria: 322
--- memoria de buffer ---
Usuario: Hola, soy Ana. Necesito un termostato para mi nevera.
Pedro: Hola Ana, ¿cuántas unidades necesitas?
Usuario: Que sean 2 unidades.
Pedro: Anotado: 2 unidades. ¿Presupuesto y plazo?
Usuario: Tengo hasta R$ 900 y lo necesito en 5 días. Resume mi solicitud.
Pedro: Ana, pediste 2 termostatos, hasta R$ 900, en 5 días.


/home/efraxpc/infnet/agentes/pedro_asistente_de_compras/src/pedro/legacy.py:121: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memoria = ConversationBufferMemory(human_prefix="Usuario", ai_prefix="Pedro")
/home/efraxpc/infnet/agentes/pedro_asistente_de_compras/.venv/lib/python3.13/site-packages/pydantic/main.py:280: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic